# Generating the results from the reduced model with derived variables, minus the delphi and household alert count

In [1]:
# Opening the datasets

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)

# Locate the project folder.
relative_folder = Path("data") / "KGB Modelling"

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_folder / "KGB in sample.csv").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the KGB modelling data.")

data_folder = project_root / relative_folder

# Load the development and validation samples.
kgb_in_sample = pd.read_csv(data_folder / "KGB in sample.csv")
kgb_out_sample = pd.read_csv(data_folder / "KGB out sample.csv")

target = "target_bad"

# Check the target is complete and binary.
for name, data in [
    ("In-sample", kgb_in_sample),
    ("Out-of-sample", kgb_out_sample),
]:
    if target not in data.columns:
        raise ValueError(f"{name}: missing {target}.")

    if not data[target].isin([0, 1]).all():
        raise ValueError(f"{name}: {target} must contain only 0 and 1.")

    if data[target].nunique() != 2:
        raise ValueError(f"{name}: both good and bad accounts are required.")

# Review sample sizes and bad rates.
sample_summary = pd.DataFrame([
    {
        "Sample": name,
        "Accounts": len(data),
        "Bads": int(data[target].sum()),
        "Bad rate": data[target].mean(),
    }
    for name, data in [
        ("In-sample", kgb_in_sample),
        ("Out-of-sample", kgb_out_sample),
    ]
])

In [2]:
# Creating the variables

def add_derived_variables(data):
    data = data.copy()

    required_columns = [
        "disposable_income",
        "time_at_address_months",
        "employment_tenure_months",
    ]

    missing_columns = set(required_columns) - set(data.columns)
    if missing_columns:
        raise ValueError(f"Missing columns: {sorted(missing_columns)}")

    for column in required_columns:
        data[column] = pd.to_numeric(data[column], errors="raise")

        if data[column].isin([np.inf, -np.inf]).any():
            raise ValueError(f"{column} contains infinite values.")

    for column in [
        "time_at_address_months",
        "employment_tenure_months",
    ]:
        if data[column].dropna().lt(0).any():
            raise ValueError(f"{column} contains negative tenure.")

    # Nullable numbers preserve missing inputs in comparisons.
    surplus = data["disposable_income"].astype("Float64")
    address_tenure = data["time_at_address_months"].astype("Float64")
    employment_tenure = data["employment_tenure_months"].astype("Float64")

    # 1 = disposable income below £150; 0 = £150 or above.
    data["low_disposable_income_flag"] = surplus.lt(150).astype("Int64")

    # Natural logarithm; zero months becomes zero.
    data["log_address_tenure"] = np.log1p(address_tenure)

    # Count of employment/address tenures below 12 months.
    data["employment_address_instability_count"] = (
        employment_tenure.lt(12).astype("Int64")
        + address_tenure.lt(12).astype("Int64")
    )

    return data


# Apply identical definitions to both samples.
kgb_in_sample = add_derived_variables(kgb_in_sample)
kgb_out_sample = add_derived_variables(kgb_out_sample)

derived_columns = [
    "low_disposable_income_flag",
    "log_address_tenure",
    "employment_address_instability_count",
]


In [3]:
# Remove only the two requested predictors.
removed = [
    "DelphiScore",
    "employment_address_instability_count",
]
X_train_new = X_train.drop(columns=removed)
X_test_new = X_test.drop(columns=removed)

# Refit all remaining coefficients using the same training accounts.
new_model = sm.Logit(y_train, X_train_new).fit(
    maxiter=200, disp=False
)

if not new_model.mle_retvals.get("converged", False):
    raise RuntimeError("Model did not converge.")
if (
    not np.isfinite(new_model.params).all()
    or not np.isfinite(new_model.bse).all()
):
    raise RuntimeError("Invalid coefficients or standard errors.")

new_train_pd = new_model.predict(X_train_new)
new_test_pd = new_model.predict(X_test_new)

comparison = pd.DataFrame({
    "Original in-sample": performance(y_train, train_pd),
    "New in-sample": performance(y_train, new_train_pd),
    "Original out-of-sample": performance(y_test, test_pd),
    "New out-of-sample": performance(y_test, new_test_pd),
})

display(comparison.round(6))
print(f"Original training AIC: {reduced_model.aic:.2f}")
print(f"New training AIC: {new_model.aic:.2f}")
print(f"New training McFadden pseudo R²: {new_model.prsquared:.4f}")
print(new_model.summary())

NameError: name 'X_train' is not defined